# Dataset piloto multicaso

Objetivo: procesar varios registros de glucosa de VitalDB para construir un dataset piloto con múltiples valores de BGL.

Este notebook parte del flujo validado en `01_exploracion.ipynb`, pero ahora busca repetir el procesamiento para más de un paciente o medición.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Rutas principales del proyecto
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

PILOT_MULTI_DIR = PROCESSED_DIR / "pilot_multicase"
PILOT_MULTI_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("RAW_DIR:", RAW_DIR.resolve())
print("INTERIM_DIR:", INTERIM_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())
print("PILOT_MULTI_DIR:", PILOT_MULTI_DIR.resolve())

PROJECT_ROOT: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl
RAW_DIR: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl\data\raw
INTERIM_DIR: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl\data\interim
PROCESSED_DIR: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl\data\processed
PILOT_MULTI_DIR: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl\data\processed\pilot_multicase


In [3]:
map_path = INTERIM_DIR / "BGL_PPG_map.csv" # Ruta al archivo de mapeo

print("Map path:", map_path.resolve())

print("\nExiste el archivo de mapeo?")
print(map_path.exists())

bgl_ppg_map = pd.read_csv(map_path)

print("Forma del mapa:")
print(bgl_ppg_map.shape)

print("Columnas:")
print(bgl_ppg_map.columns.tolist())

display(bgl_ppg_map.head())

Map path: C:\Users\axxel\Desktop\Tesis\nuevo\predict_bgl\data\interim\BGL_PPG_map.csv

Existe el archivo de mapeo?
True
Forma del mapa:
(29713, 9)
Columnas:
['caseid', 'tname', 'ppg_track_tid', 'bgl_timestamp_sec', 'name', 'bgl_value_mg_dL', 'age', 'sex', 'bmi']


,caseid,tname,ppg_track_tid,bgl_timestamp_sec,name,bgl_value_mg_dL,age,sex,bmi
0,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,3060,gluc,154.0,77.0,M,26.3
1,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,4628,gluc,182.0,77.0,M,26.3
2,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,12614,gluc,198.0,77.0,M,26.3
3,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,8921,gluc,210.0,77.0,M,26.3
4,2,SNUADC/PLETH,8dc8fb6a62bf86f5117e4b5ea679f061c30b8640,5857898,gluc,114.0,54.0,M,19.6


In [4]:
bgl_ppg_map = bgl_ppg_map.rename(columns={
    "ppg_track_tid" : "ppg_track_id"
    })

In [5]:
print("Valores nulos en mapa:")
print(bgl_ppg_map.isnull().sum())

print("Números de caseid únicos")
print(bgl_ppg_map["caseid"].nunique())

print("Valores de glucosa únicos")
print(bgl_ppg_map["bgl_value_mg_dL"].nunique())

Valores nulos en mapa:
caseid               0
tname                0
ppg_track_id         0
bgl_timestamp_sec    0
name                 0
bgl_value_mg_dL      0
age                  0
sex                  0
bmi                  0
dtype: int64
Números de caseid únicos
4872
Valores de glucosa únicos
421


In [18]:
#Revisar cuántas mediciones tiene cada caso

medidas_por_caso = (
    bgl_ppg_map.groupby("caseid")
    .size()
    .reset_index(name="n_bgl_records")
    .sort_values("n_bgl_records", ascending=False)
    .reset_index(drop=True)
)

print("Forma de las medidas por caso:")
print(medidas_por_caso.shape)

print("\nPrimeros pacientes con más mediciones")
display(medidas_por_caso.head(10))

print("\nResumen de número de mediciones por paciente")
display(medidas_por_caso["n_bgl_records"].describe())

Forma de las medidas por caso:
(4872, 2)

Primeros pacientes con más mediciones


,caseid,n_bgl_records
0,6075,111
1,1448,111
2,3438,111
3,4748,111
4,4647,107
5,6337,107
6,4898,106
7,813,104
8,5691,90
9,846,89



Resumen de número de mediciones por paciente


count    4872.000000
mean        6.098727
std         8.713081
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max       111.000000
Name: n_bgl_records, dtype: float64

In [27]:
#Seleccionar 5 registros distintos
pilot_records = (
    bgl_ppg_map
    .sort_values(["caseid", "bgl_timestamp_sec"])
    .groupby("caseid", as_index=False)
    .first()
    .head()
)

print("Forma de registros seleccionados:")
display(pilot_records.shape)

print("\nRegistros seleccionados:")
display(
    pilot_records[
        [
            "caseid",
            "tname",
            "ppg_track_id",
            "bgl_timestamp_sec",
            "bgl_value_mg_dL",
            "age",
            "sex",
            "bmi"
        ]
    ]
)

print("\nValores únicos de glucosa en los registros pilotos:")
print(pilot_records["bgl_value_mg_dL"].unique())

Forma de registros seleccionados:


(5, 9)


Registros seleccionados:


,caseid,tname,ppg_track_id,bgl_timestamp_sec,bgl_value_mg_dL,age,sex,bmi
0,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,3060,154.0,77.0,M,26.3
1,2,SNUADC/PLETH,8dc8fb6a62bf86f5117e4b5ea679f061c30b8640,152248,118.0,54.0,M,19.6
2,4,SNUADC/PLETH,e8d08e829a45456e8890f47d0995f1b3064529b4,3078,108.0,74.0,M,20.5
3,5,SNUADC/PLETH,7cfa86c1b838540f2956240169a83f3723b4092a,-2968,140.0,66.0,M,20.4
4,7,SNUADC/PLETH,9f042039ebdd72b489b1bdcc03b3affe29bc3eb6,-97335,94.0,52.0,F,22.2



Valores únicos de glucosa en los registros pilotos:
[154. 118. 108. 140.  94.]


In [28]:
#Como algunos registros tienen time negativo, nos aseguramos que sean mayores a 480 s (ya que las ventanas son 8 min hacia atrás y 8 min hacia adelante)
valid_map = bgl_ppg_map[
    bgl_ppg_map["bgl_timestamp_sec"] >= 480
].copy()

pilot_records = (
    valid_map
    .sort_values(["caseid", "bgl_timestamp_sec"])
    .groupby("caseid", as_index=False)
    .first()
    .head(5)
)

print("Forma de valid_map:", valid_map.shape)
print("Forma de registros seleccionados:", pilot_records.shape)

display(
    pilot_records[
        [
            "caseid",
            "tname",
            "ppg_track_id",
            "bgl_timestamp_sec",
            "bgl_value_mg_dL",
            "age",
            "sex",
            "bmi"
        ]
    ]
)

print("\nValores únicos de glucosa en los registros piloto:")
print(pilot_records["bgl_value_mg_dL"].unique())

print("\nTimestamp mínimo seleccionado:")
print(pilot_records["bgl_timestamp_sec"].min())

Forma de valid_map: (21618, 9)
Forma de registros seleccionados: (5, 9)


,caseid,tname,ppg_track_id,bgl_timestamp_sec,bgl_value_mg_dL,age,sex,bmi
0,1,SNUADC/PLETH,9acbed98f1f15c7827ee3bcc55eaef19f861b824,3060,154.0,77.0,M,26.3
1,2,SNUADC/PLETH,8dc8fb6a62bf86f5117e4b5ea679f061c30b8640,152248,118.0,54.0,M,19.6
2,4,SNUADC/PLETH,e8d08e829a45456e8890f47d0995f1b3064529b4,3078,108.0,74.0,M,20.5
3,5,SNUADC/PLETH,7cfa86c1b838540f2956240169a83f3723b4092a,7172,89.0,66.0,M,20.4
4,7,SNUADC/PLETH,9f042039ebdd72b489b1bdcc03b3affe29bc3eb6,4092,103.0,52.0,F,22.2



Valores únicos de glucosa en los registros piloto:
[154. 118. 108.  89. 103.]

Timestamp mínimo seleccionado:
3060
